# XGBoost Hyperparameter Tuning

This experiment tests several XGBoost configurations using the same validation split as the previous experiments.

The goal is to improve on the previous XGBoost ROC-AUC of **0.941635** without changing the dataset or validation setup.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

# Find the project root from the notebook's working directory.
current_path = Path.cwd().resolve()
project_root = current_path

while project_root.name != 'DataCompetition' and project_root.parent != project_root:
    project_root = project_root.parent

data_path = project_root / 'data' / 'train.csv'

if not data_path.exists():
    raise FileNotFoundError(f'Could not find training data at: {data_path}')

print('Project root:', project_root)
print('Training data:', data_path)

current = Path.cwd().resolve()
project_root = None

for folder in [current, *current.parents]:
    if folder.name == "DataCompetition" and (folder / "data" / "train.csv").exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find the DataCompetition project folder containing data/train.csv."
    )

train_path = project_root / "data" / "train.csv"
train = pd.read_csv(train_path)

print(f"Project root: {project_root}")
print(f"Train data: {train_path}")
print(f"Train shape: {train.shape})

X = train.drop(columns=['Will_Buy_EV', 'id'])
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

print('Training rows:', X_train.shape[0])
print('Validation rows:', X_valid.shape[0])
print('Processed features:', X_train_processed.shape[1])

## Model configurations

We will test different combinations of tree depth, learning rate, number of trees, and regularization.

All models use the same training and validation data.

In [ ]:
configs = [
    {
        'name': 'XGB T1 - deeper',
        'n_estimators': 700,
        'max_depth': 7,
        'learning_rate': 0.04,
        'min_child_weight': 1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0,
        'reg_alpha': 0,
        'reg_lambda': 1
    },
    {
        'name': 'XGB T2 - shallower',
        'n_estimators': 800,
        'max_depth': 5,
        'learning_rate': 0.04,
        'min_child_weight': 1,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'gamma': 0,
        'reg_alpha': 0,
        'reg_lambda': 1
    },
    {
        'name': 'XGB T3 - regularized',
        'n_estimators': 700,
        'max_depth': 6,
        'learning_rate': 0.04,
        'min_child_weight': 3,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 2
    },
    {
        'name': 'XGB T4 - balanced',
        'n_estimators': 900,
        'max_depth': 6,
        'learning_rate': 0.03,
        'min_child_weight': 1,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'gamma': 0,
        'reg_alpha': 0,
        'reg_lambda': 1
    },
    {
        'name': 'XGB T5 - stronger regularization',
        'n_estimators': 900,
        'max_depth': 6,
        'learning_rate': 0.03,
        'min_child_weight': 3,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'gamma': 0.1,
        'reg_alpha': 0.2,
        'reg_lambda': 3
    }
]

results = []

for config in configs:
    model = XGBClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        learning_rate=config['learning_rate'],
        min_child_weight=config['min_child_weight'],
        subsample=config['subsample'],
        colsample_bytree=config['colsample_bytree'],
        gamma=config['gamma'],
        reg_alpha=config['reg_alpha'],
        reg_lambda=config['reg_lambda'],
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_processed, y_train)
    predictions = model.predict_proba(X_valid_processed)[:, 1]
    auc = roc_auc_score(y_valid, predictions)

    results.append({
        'Model': config['name'],
        'ROC-AUC': auc
    })

    print(f"{config['name']}: {auc:.6f}")

In [ ]:
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print('\nTuning results:')
display(results_df)

best_model = results_df.iloc[0]

print(f"\nBest configuration: {best_model['Model']}")
print(f"Best ROC-AUC: {best_model['ROC-AUC']:.6f}")
print('Previous XGBoost ROC-AUC: 0.941635')
print(f"Improvement: {best_model['ROC-AUC'] - 0.941635:+.6f}")

## Current experiment history

| Model | Validation ROC-AUC |
|---|---:|
| Logistic Regression | 0.938000 |
| HistGradientBoosting | 0.940597 |
| XGBoost | 0.941635 |
| XGBoost + Feature Engineering | 0.941344 |
| XGBoost Hyperparameter Tuning | See results above |